# **Fooling LIME and SHAP: building a scaffolded model**

Practice for the module ["Attacks on Explanations"](https://ai-interpretability.school).

In the lesson we covered the attack of [Slack et al. (2020)](https://arxiv.org/abs/1911.02508): a model that uses race on real inputs but behaves innocuously on the synthetic perturbations with which LIME and KernelSHAP query it. Here we build the whole attack on the open UCI Adult data, in the same setup as the paper (the authors showed it on COMPAS, among other datasets):

- the biased model $f$ decides by race alone;
- the innocuous model $\psi$ decides by a random column `unrelated_column` that is unrelated to everything;
- a perturbation detector decides which of the two models to show;
- the scaffolded model is $e(x)=f(x)$ if the detector considers $x$ a real input, and $e(x)=\psi(x)$ otherwise.

By the end of the notebook you will have answers to three questions: how closely the scaffold matches $f$ on real data, what LIME and SHAP see, and whether the attack transfers from one explainer to the other. Exercises 4 and 5 are about how to catch such a model, and exercises 6–8 are for the closing lesson of the module.

The notebook runs on a CPU in one or two minutes.

In [ ]:
!pip install lime==0.2.0.1 shap==0.52.0 -q

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import lime.lime_tabular
import shap
import warnings
warnings.filterwarnings('ignore')   # numpy and tqdm print harmless warnings on some platforms
from sklearn.cluster import KMeans
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

SEED = 0
rng = np.random.default_rng(SEED)   # every random step of the notebook draws from this generator

## The UCI Adult data

We take the training part of the Adult dataset from the UCI Machine Learning Repository (Becker and Kohavi, 1996, license CC BY 4.0, DOI [10.24432/C5XW20](https://doi.org/10.24432/C5XW20)): 32561 people from the 1994 US Census. The file `data/adult.csv` in the course repository is `adult.data` with a header row of column names and without the spaces after commas. The target is an annual income above 50 thousand dollars; the attack does not need it, because the biased model ignores it. The sensitive feature is `race_black`: 1 for people recorded in the census with race Black, 9.6% of them. The feature `married` is 1 for the marital status Married-civ-spouse or Married-AF-spouse. The column `unrelated_column` is random 0s and 1s that we add ourselves. All features are standardized: LIME perturbs the data with Gaussian noise on this scale.

In [ ]:
URL = 'https://raw.githubusercontent.com/SadSabrina/open-xai-materials/main/data/adult.csv'
df = pd.read_csv(URL)

X = pd.DataFrame({
    'age': df.age,
    'education_num': df['education-num'],
    'hours_per_week': df['hours-per-week'],
    'capital_gain': df['capital-gain'],
    'married': df['marital-status'].isin(['Married-civ-spouse', 'Married-AF-spouse']).astype(int),
    'sex_male': (df.sex == 'Male').astype(int),
    'race_black': (df.race == 'Black').astype(int),
    'unrelated_column': rng.integers(0, 2, len(df)),
})
y = (df.income == '>50K').astype(int).values

features = list(X.columns)
RACE, UNREL = features.index('race_black'), features.index('unrelated_column')
X_train, X_test, y_train, y_test = train_test_split(X.values.astype(float), y, test_size=0.1, random_state=SEED)
scaler = StandardScaler().fit(X_train)
X_train, X_test = scaler.transform(X_train), scaler.transform(X_test)
print(len(X), 'inputs, share Black:', round(X.race_black.mean(), 3))
print('train / test:', len(X_train), '/', len(X_test))

## Two models: biased and innocuous

Both models are deliberately primitive, which makes it easy to see which of them the explainer sees. On the standardized scale, `race_black` is positive exactly for people with this label, and `unrelated_column` is positive exactly where it holds a 1.

In [ ]:
def f(Z):
    """Biased model: uses race and nothing else."""
    return (Z[:, RACE] > 0).astype(int)

def psi(Z):
    """Innocuous model: uses a random column that is unrelated to everything."""
    return (Z[:, UNREL] > 0).astype(int)

def as_proba(predict):
    """LIME and SHAP want class probabilities, not labels."""
    return lambda Z: np.c_[1 - predict(Z), predict(Z)]

## A perturbation detector against LIME and the scaffolded model

LIME (without discretization) builds the neighbors of an input by adding $\mathcal N(0,1)$ noise to the features. We train a random forest to tell training inputs (label 0) from their noisy copies (label 1), exactly as the paper describes the detector. The scaffolded model asks the detector and chooses whom to be.

In [ ]:
X_pert = X_train + rng.normal(0, 1, X_train.shape)   # how LIME perturbs standardized data
ood_lime = RandomForestClassifier(n_estimators=100, random_state=SEED).fit(
    np.vstack([X_train, X_pert]), np.r_[np.zeros(len(X_train)), np.ones(len(X_pert))])

def scaffold(ood):
    """e(x) = f(x) on real inputs, psi(x) on inputs the detector calls perturbations."""
    return lambda Z: np.where(ood.predict(Z) == 1, psi(Z), f(Z))

e_lime = scaffold(ood_lime)
print('agreement of e_lime with f on test inputs:', (e_lime(X_test) == f(X_test)).mean())

## What LIME sees

We explain the first 100 test inputs and for each one take the feature with the largest absolute weight. We print the share of the 100 inputs that have race in first place and the share that have `unrelated_column` there.

In [ ]:
explainer = lime.lime_tabular.LimeTabularExplainer(
    X_train, feature_names=features, discretize_continuous=False, random_state=SEED)
N = 100   # explain the first 100 test inputs

def lime_top1(predict):
    """Index of the feature with the largest |weight| in the LIME explanation, per input."""
    tops = []
    for x in X_test[:N]:
        exp = explainer.explain_instance(x, as_proba(predict), num_features=len(features))
        tops.append(max(exp.as_map()[1], key=lambda t: abs(t[1]))[0])
    return np.array(tops)

lime_f, lime_e = lime_top1(f), lime_top1(e_lime)
for name, tops in [('f', lime_f), ('e_lime', lime_e)]:
    print(f'LIME, {name:7s}: race {(tops == RACE).mean():.2f}, unrelated_column {(tops == UNREL).mean():.2f}')

## A perturbation detector against KernelSHAP

KernelSHAP perturbs an input differently: it keeps some features and takes the rest from a point of the background set. The background is 10 k-means centers, as in the paper. A detector trained on Gaussian noise does not recognize such points, so for SHAP we train a separate detector on KernelSHAP-style perturbations.

In [ ]:
background = KMeans(n_clusters=10, random_state=SEED, n_init=10).fit(X_train).cluster_centers_

def shap_perturb(Xs):
    """KernelSHAP-style perturbation: a random subset of features takes values from a background point."""
    P = Xs.copy()
    for i in range(len(P)):
        mask = rng.random(P.shape[1]) < 0.5
        P[i, mask] = background[rng.integers(0, len(background)), mask]
    return P

X_shap_pert = shap_perturb(X_train)
ood_shap = RandomForestClassifier(n_estimators=100, random_state=SEED).fit(
    np.vstack([X_train, X_shap_pert]), np.r_[np.zeros(len(X_train)), np.ones(len(X_shap_pert))])
e_shap = scaffold(ood_shap)

## What SHAP sees

The same 100 inputs, KernelSHAP with 500 coalitions per input. The seed is fixed, so a rerun gives the same numbers.

In [ ]:
def shap_top1(predict):
    """Index of the feature with the largest |SHAP value|, per input."""
    np.random.seed(SEED)   # KernelSHAP samples coalitions with np.random
    ke = shap.KernelExplainer(lambda Z: predict(Z).astype(float), background)
    values = ke.shap_values(X_test[:N], nsamples=500, silent=True)
    return np.abs(values).argmax(axis=1)

shap_f, shap_e = shap_top1(f), shap_top1(e_shap)
for name, tops in [('f', shap_f), ('e_shap', shap_e)]:
    print(f'SHAP, {name:7s}: race {(tops == RACE).mean():.2f}, unrelated_column {(tops == UNREL).mean():.2f}')

## Figure: which feature the explainer ranks first

Four panels: the biased model and the scaffold under LIME and under SHAP. Bar height is the share of the 100 test inputs that have this feature in first place.

In [ ]:
panels = [('LIME, f', lime_f), ('LIME, e_lime', lime_e), ('SHAP, f', shap_f), ('SHAP, e_shap', shap_e)]
fig, axes = plt.subplots(1, 4, figsize=(15, 3.4), sharey=True)
for ax, (title, tops) in zip(axes, panels):
    shares = [(tops == RACE).mean(), (tops == UNREL).mean(), 1 - (tops == RACE).mean() - (tops == UNREL).mean()]
    ax.bar(['race', 'unrelated', 'other'], shares, color=['#eb6834', '#2a78d6', '#8a8983'])
    ax.set_title(title)
    ax.set_ylim(0, 1.05)
axes[0].set_ylabel('share of 100 test inputs\nwith this feature ranked 1st')
plt.tight_layout()
plt.show()

**Exercise 1.** The scaffold against SHAP fools the explainer, but on real data it must also make the same decisions as $f$, otherwise an ordinary accuracy check would expose the fake. On how many test inputs (out of all test inputs) do `e_shap` and `f` give different predictions? The answer is an integer.

In [ ]:
# Your code here

**Exercise 2.** From the output of the cell "What SHAP sees": for what share of the 100 explained inputs does SHAP still rank race first for `e_shap`? The answer is a decimal with two digits.

**Exercise 3.** The scaffold `e_lime` was built against LIME. Explain it with KernelSHAP (the function `shap_top1` is already written) and compute the share of the 100 inputs that have race in first place. Then check what share of SHAP perturbations of the test inputs (`shap_perturb(X_test)`) the detector `ood_lime` flags as perturbations. How do the two numbers explain each other?

In [ ]:
# Your code here

**Exercise 4. How to catch the scaffold.** An auditor with black-box access to the model can compare it not only on real inputs but also on their perturbed copies. Add $\mathcal N(0,1)$ noise to the test inputs (use `rng.normal`, as above) and compute the share of perturbed copies on which `e_lime` and `f` give different predictions. Compare it with the same share on the test inputs themselves. The answer is the share on the perturbed copies, three decimal places.

*Hint:* of course the auditor does not know the biased model `f`; here `f` is needed to make the gap visible. In practice the sign of a fake is that the model behaves on the explainer's perturbations unlike on the data.

In [ ]:
# Your code here

**Exercise 5.** In two or three lines: why is the share from exercise 4 close to one half? Which defense from the lesson does it suggest?

**Your answer:**

## Additional exercises

Three exercises for the closing lesson of the module. They continue exercises 3 and 4: does the scaffold against SHAP transfer to LIME, and how often does each perturbation detector err.

**Exercise 6.** The scaffold `e_shap` was built against KernelSHAP. Explain it with LIME and compute the share of the 100 inputs that have race in first place. For this, create a new `LimeTabularExplainer` with the same parameters and `random_state=SEED`: then the number does not depend on how many times the cells above were run. The answer is a decimal with two digits.

In [ ]:
# Your code here

**Exercise 7.** What share of Gaussian perturbations of the test inputs ($\mathcal N(0,1)$ noise, as in LIME) does the detector `ood_shap` flag as perturbations? Take the noise from a separate generator `np.random.default_rng(1)` so that `rng` is not shifted. The answer has three decimal places.

In [ ]:
# Your code here

**Exercise 8.** The detector also errs the other way: it takes a real input for a perturbation. What share of the 3257 test inputs does the detector `ood_shap` flag as perturbations? Compare with the answer to exercise 1: why are there fewer inputs on which `e_shap` and `f` disagree? The answer has three decimal places.

In [ ]:
# Your code here

## What to take away

- **The attack needs no access to the explainer, only knowledge of how it builds neighbors.** The perturbation detector is an ordinary random forest, and the scaffold matches the biased model on the real test inputs.
- **On the 100 explained inputs both LIME and SHAP are fooled: race is ranked first for none of them.** The detector against SHAP still errs: it takes 18 of the 3257 real test inputs for perturbations, and on 10 of them the scaffold disagrees with the biased model (exercises 1 and 8). In the paper too, the scaffold against SHAP hides the feature on most but not all inputs.
- **The attack is tied to the perturbation scheme.** The scaffold against LIME does not recognize KernelSHAP perturbations, and SHAP sees race.
- **The sign of a fake is a mismatch between behavior on data and on perturbations.** Hence the defenses from the lesson: perturb closer to the data manifold and check whether the model behaves on the explainer's neighbors differently than on the data.